<a href="https://colab.research.google.com/github/suleiman-odeh/NLP_Project_Team16/blob/main/fine_tuning/fine_tuning_indirect_Qwen2_5_7B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"


!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install scikit-learn pandas tqdm imbalanced-learn

  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-install-pvi117zh/unsloth-zoo_c7b671d3ea9b46cbb430696aa101d803
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth-zoo.git /tmp/pip-install-pvi117zh/unsloth-zoo_c7b671d3ea9b46cbb430696aa101d803
  Resolved https://github.com/unslothai/unsloth-zoo.git to commit ba585aff2b3f80594497171c8d6d216f921cdb8d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7nlojtff/unsloth_f7ea5fd1e091460d86254991b26c1396
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7nlojtff/unsloth_f7ea5fd1e091460d86254991b26c1396
  Resolved https://github.com/unslothai/unsloth.git to commit a50b4337ae526744711f54f9aed6e82c778280fd
  Installing build dependencies ... done
  Getting requirements to buil

In [ ]:
import os
import torch
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from collections import Counter
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report
import gc

BASE_PATH = "/content/drive/MyDrive/NLP_Project_QEvasion"
DATA_FILE = os.path.join(BASE_PATH, "QEvasion_cleaned.jsonl")
MODELS_DIR = os.path.join(BASE_PATH, "models_paper_replication")
os.makedirs(MODELS_DIR, exist_ok=True)

# Model Config
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

# Loading data
def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found. Please upload it to Drive.")
    try:
        df = pd.read_json(file_path, lines=True)
    except ValueError:
        df = pd.read_json(file_path)
    df = df.dropna(how='all')
    train_df = df[df['split_type'] == 'train'].copy()
    test_df = df[df['split_type'] == 'test'].copy()
    if len(test_df) == 0:
        test_df = df[df['split_type'] == 'dev'].copy()
    return train_df, test_df

train_df, test_df = load_data(DATA_FILE)

# Mappings
evasion_label_map = {
    0: "Explicit", 1: "Implicit", 2: "General", 3: "Partial",
    4: "Dodging", 5: "Deflection", 6: "Declining to answer",
    7: "Claims ignorance", 8: "Clarification"
}
clarity_label_map = {0: "Clear Reply", 1: "Ambivalent", 2: "Clear Non-Reply"}
evasion_to_clarity_id = {0:0, 1:1, 2:1, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2}

def get_evasion_text(row): return evasion_label_map.get(row.get('evasion_id'), "Unknown")
def get_clarity_text(row):
    cid = row.get('clarity_id')
    if pd.isna(cid) and pd.notna(row.get('evasion_id')): cid = evasion_to_clarity_id.get(row.get('evasion_id'))
    return clarity_label_map.get(cid, "Unknown")

train_df['evasion_text'] = train_df.apply(get_evasion_text, axis=1)
train_df['clarity_text'] = train_df.apply(get_clarity_text, axis=1)

print(f"Data Loaded. Train: {len(train_df)} | Test: {len(test_df)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Using MoE backend 'grouped_mm'
🦥 Unsloth Zoo will now patch everything to make training faster!
Data Loaded. Train: 3448 | Test: 308


In [ ]:
# 1. Get Tokenizer for EOS token
model_id_temp = "unsloth/llama-2-7b-bnb-4bit"
tokenizer_temp = FastLanguageModel.from_pretrained(model_id_temp, load_in_4bit=True)[1]
EOS_TOKEN = tokenizer_temp.eos_token

# EVASION DATASETS
def format_evasion(sample):
    # YOUR PROMPT
    instruction = (
        "Based on a part of the interview where the interviewer asks a set of questions, "
        "classify the type of answer the interviewee provided for the following question "
        "into one of these evasion categories:\n"
        "1. Explicit\n2. Implicit\n3. General\n4. Partial\n"
        "5. Dodging\n6. Deflection\n7. Declining to answer\n"
        "8. Claims ignorance\n9. Clarification"
    )

    prompt = f"{instruction}\n\n### Part of the interview ###\n{sample['cleaned_answer']}\n\n### Question ###\n{sample['question']}\n\nLabel: "

    return { "text": f"{prompt}{sample['evasion_text']}{EOS_TOKEN}" }

# Split
ds_ev = Dataset.from_pandas(train_df)
split_ev = ds_ev.train_test_split(test_size=0.2175, seed=42)
train_ev_std = split_ev['train'].map(format_evasion)
eval_ev = split_ev['test'].map(format_evasion)

# CLARITY DATASETS
def format_clarity(sample):
    instruction = (
        "Based on a part of the interview where the interviewer asks a set of questions, "
        "classify the type of answer the interviewee provided for the following question "
        "into one of these categories:\n"
        "1. Clear Reply\n"
        "2. Clear Non-Reply\n"
        "3. Ambivalent"
    )

    prompt = f"{instruction}\n\n### Part of the interview ###\n{sample['cleaned_answer']}\n\n### Question ###\n{sample['question']}\n\nLabel: "

    return { "text": f"{prompt}{sample['clarity_text']}{EOS_TOKEN}" }

# Split
ds_cl = Dataset.from_pandas(train_df)
split_cl = ds_cl.train_test_split(test_size=0.2175, seed=42)
train_cl_std = split_cl['train'].map(format_clarity)
eval_cl = split_cl['test'].map(format_clarity)

print("Datasets Ready (Standard Distribution).")

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Map:   0%|          | 0/2698 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/2698 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Datasets Ready (Standard Distribution).


In [ ]:
def run_experiment(model_id, experiment_name, train_dataset, eval_dataset, epochs=5, batch_size=4, grad_accum=4):
    output_dir = os.path.join(MODELS_DIR, experiment_name)
    print(f"\n{'='*40}")
    print(f"STARTING: {experiment_name}")
    print(f"{'='*40}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_id,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = DTYPE,
        load_in_4bit = LOAD_IN_4BIT,
    )

    # Parameters
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 32,
        lora_dropout = 0.05,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        dataset_num_proc = 2,
        packing = False,
        args = TrainingArguments(
            per_device_train_batch_size = batch_size,
            gradient_accumulation_steps = grad_accum,
            warmup_steps = 10,
            num_train_epochs = epochs,
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = os.path.join(output_dir, "checkpoints"),
            save_strategy = "no",
            report_to = "none"
        ),
    )

    trainer.train()

    print(f"Saving to {output_dir}...")
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
experiments = [
    # 1. Evasion
    {"id": "unsloth/llama-2-7b-bnb-4bit", "name": "Llama7B-Evasion-Std", "ds": train_ev_std, "eval": eval_ev},

    # 2. Clarity
    {"id": "unsloth/llama-2-7b-bnb-4bit", "name": "Llama7B-Clarity-Std", "ds": train_cl_std, "eval": eval_cl},
]

for exp in experiments:
    run_experiment(
        model_id = exp["id"],
        experiment_name = exp["name"],
        train_dataset = exp["ds"],
        eval_dataset = exp["eval"],
        epochs = 5,
        batch_size = 4,
        grad_accum = 4
    )


STARTING: Llama7B-Evasion-Std
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.2.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2698 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/750 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,698 | Num Epochs = 5 | Total steps = 845
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)


...

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2698 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/750 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,698 | Num Epochs = 5 | Total steps = 845
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)


Step,Training Loss
10,2.124000
20,1.681100
30,1.708000
40,1.639900
50,1.667100
60,1.643000
70,1.643000
80,1.610100
90,1.592200
100,1.648800


Saving to /content/drive/MyDrive/NLP_Project_QEvasion/models_paper_replication/Llama7B-Clarity-Std...


In [ ]:
# Evaluation
def evaluate_evasion(adapter_path, dataset_df):
    print(f"\nEVALUATING EVASION: {adapter_path}")
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=adapter_path, max_seq_length=MAX_SEQ_LENGTH, dtype=DTYPE, load_in_4bit=LOAD_IN_4BIT)
    FastLanguageModel.for_inference(model)
    text_to_id = {v: k for k, v in evasion_label_map.items()}

    instruction = (
        "Based on a part of the interview where the interviewer asks a set of questions, "
        "classify the type of answer the interviewee provided for the following question "
        "into one of these evasion categories:\n"
        "1. Explicit\n2. Implicit\n3. General\n4. Partial\n"
        "5. Dodging\n6. Deflection\n7. Declining to answer\n"
        "8. Claims ignorance\n9. Clarification"
    )

    y_true_ev, y_pred_ev = [], []
    y_true_cl, y_pred_cl = [], []

    for index, row in tqdm(dataset_df.iterrows(), total=len(dataset_df)):
        prompt = f"{instruction}\n\n### Part of the interview ###\n{row['cleaned_answer']}\n\n### Question ###\n{row['question']}\n\nLabel: "

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        outputs = model.generate(**inputs, max_new_tokens=10, use_cache=True, do_sample=False, pad_token_id=tokenizer.eos_token_id)

        # Parse after "Label: "
        pred_text = tokenizer.batch_decode(outputs)[0].split("Label: ")[-1].replace(tokenizer.eos_token, "").replace("<|endoftext|>", "").strip().split('\n')[0]
        clean_pred = pred_text.strip().rstrip('.').rstrip(',').strip()

        pred_id = -1
        if clean_pred in text_to_id: pred_id = text_to_id[clean_pred]
        else:
             for k, v in text_to_id.items():
                if k.lower() == clean_pred.lower(): pred_id = v; break

        # Fallback for numbers
        if pred_id == -1 and clean_pred.isdigit():
             try:
                v = int(clean_pred)
                if 0 <= v-1 <= 8: pred_id = v-1
             except: pass

        # Ground Truth
        votes = []
        for c in ['annotator1_id', 'annotator2_id', 'annotator3_id']:
            if pd.notna(row.get(c)):
                try:
                    v = int(float(row[c]))
                    if 0 <= v <= 8: votes.append(v)
                except: pass
        if not votes: continue
        gold_id = Counter(votes).most_common(1)[0][0]

        if pred_id != -1:
            y_true_ev.append(gold_id)
            y_pred_ev.append(pred_id)
            y_true_cl.append(evasion_to_clarity_id[gold_id])
            y_pred_cl.append(evasion_to_clarity_id[pred_id])

    print("--- Evasion Report (9 Classes) ---")
    names = [evasion_label_map[i] for i in range(9)]
    print(classification_report(y_true_ev, y_pred_ev, labels=range(9), target_names=names, zero_division=0))

    print("--- Derived Clarity Report (Task 2 from Paper) ---")
    cl_names = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
    print(classification_report(y_true_cl, y_pred_cl, labels=range(3), target_names=cl_names, zero_division=0))

    del model; torch.cuda.empty_cache()


def evaluate_clarity(adapter_path, dataset_df):
    print(f"\nEVALUATING CLARITY: {adapter_path}")
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=adapter_path, max_seq_length=MAX_SEQ_LENGTH, dtype=DTYPE, load_in_4bit=LOAD_IN_4BIT)
    FastLanguageModel.for_inference(model)
    text_to_id = {v: k for k, v in clarity_label_map.items()}

    instruction = (
        "Based on a part of the interview where the interviewer asks a set of questions, "
        "classify the type of answer the interviewee provided for the following question "
        "into one of these categories:\n"
        "1. Clear Reply\n"
        "2. Clear Non-Reply\n"
        "3. Ambivalent"
    )

    y_true, y_pred = [], []

    for index, row in tqdm(dataset_df.iterrows(), total=len(dataset_df)):
        prompt = f"{instruction}\n\n### Part of the interview ###\n{row['cleaned_answer']}\n\n### Question ###\n{row['question']}\n\nLabel: "

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=10, use_cache=True, do_sample=False, pad_token_id=tokenizer.eos_token_id)

        pred_text = tokenizer.batch_decode(outputs)[0].split("Label: ")[-1].replace(tokenizer.eos_token, "").replace("<|endoftext|>", "").strip().split('\n')[0]
        clean_pred = pred_text.strip().rstrip('.').rstrip(',').strip()

        pred_id = -1
        if clean_pred in text_to_id: pred_id = text_to_id[clean_pred]
        else:
             for k, v in text_to_id.items():
                if k.lower() == clean_pred.lower(): pred_id = v; break

        # Fallback
        if pred_id == -1 and clean_pred.isdigit():
             try:
                v = int(clean_pred)
                if 0 <= v-1 <= 2: pred_id = v-1
             except: pass

        votes = []
        for c in ['annotator1_id', 'annotator2_id', 'annotator3_id']:
            if pd.notna(row.get(c)):
                try:
                    v = int(float(row[c]))
                    if 0 <= v <= 8: votes.append(evasion_to_clarity_id[v])
                except: pass
        if not votes: continue
        gold_id = Counter(votes).most_common(1)[0][0]

        if pred_id != -1:
            y_true.append(gold_id)
            y_pred.append(pred_id)

    names = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
    print(classification_report(y_true, y_pred, labels=range(3), target_names=names, zero_division=0))
    del model; torch.cuda.empty_cache()

evaluate_evasion(os.path.join(MODELS_DIR, "Llama7B-Evasion-Std"), test_df)
evaluate_clarity(os.path.join(MODELS_DIR, "Llama7B-Clarity-Std"), test_df)


EVALUATING EVASION: /content/drive/MyDrive/NLP_Project_QEvasion/models_paper_replication/Llama7B-Evasion-Std
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


  0%|          | 0/308 [00:00<?, ?it/s]

--- Evasion Report (9 Classes) ---
                     precision    recall  f1-score   support

           Explicit       0.03      0.25      0.05         8
           Implicit       0.00      0.00      0.00         4
            General       0.00      0.00      0.00        12
            Partial       0.00      0.00      0.00        19
            Dodging       0.67      0.08      0.14        51
         Deflection       0.32      0.71      0.44        86
Declining to answer       0.00      0.00      0.00        47
   Claims ignorance       0.00      0.00      0.00        54
      Clarification       0.00      0.00      0.00         3

           accuracy                           0.24       284
          macro avg       0.11      0.12      0.07       284
       weighted avg       0.22      0.24      0.16       284

--- Derived Clarity Report (Task 2 from Paper) ---
                 precision    recall  f1-score   support

    Clear Reply       0.03      0.25      0.05         8
   

  0%|          | 0/308 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

    Clear Reply       0.00      0.00      0.00         6
     Ambivalent       0.56      0.28      0.38       145
Clear Non-Reply       0.21      0.04      0.07        98

       accuracy                           0.18       249
      macro avg       0.26      0.11      0.15       249
   weighted avg       0.41      0.18      0.25       249

